In [ ]:
!pip install -q sentence-transformers==2.7.0 faiss-cpu pypdf


In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from pypdf import PdfReader

reader = PdfReader("Machine Learning Notes.pdf")

text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text

print(text)

In [ ]:
chunk_size = 500

chunks = []

for i in range(0, len(text), chunk_size):
    chunks.append(text[i:i+chunk_size])

print("Number of Chunks:", len(chunks))

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding Model Loaded Successfully!")

In [ ]:
print(model)

In [ ]:
embeddings = model.encode(chunks)

print("Embedding Shape:", embeddings.shape)

In [ ]:
import faiss
import numpy as np

embedding_array = np.array(embeddings).astype("float32")

index = faiss.IndexFlatL2(embedding_array.shape[1])

index.add(embedding_array)

print("FAISS Index Created Successfully!")

In [ ]:
question = input("Enter your question: ")

print(question)

In [ ]:
question_embedding = model.encode([question])

D, I = index.search(
    np.array(question_embedding).astype("float32"),
    k=1
)

context = chunks[I[0][0]]

print("Retrieved Context:\n")
print(context)

In [ ]:
!pip install -q transformers

In [ ]:
from transformers import pipeline

qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad"
)

print("QA Model Loaded Successfully!")

In [ ]:
result = qa_pipeline(
    question=question,
    context=context
)

print("\nQuestion:")
print(question)

print("\nAnswer:")
print(result["answer"])